In [11]:
import pandas as pd
import os
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay, classification_report
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import torch

In [2]:
# load 21k labeled data
df_test = pd.read_parquet('/home/jko/ssl-cpi-analysis/data/labeled_data_geo_features.parquet')
df_test.head()

,classification,filename,aspect_ratio,aspect_ratio_elip,extreme_points,contour_area,contour_perimeter,filled_circular_area_ratio,complexity,circularity,roundness,perim_area_ratio,solidity,equiv_d,laplacian
0,compact_irreg,/home/vanessa/hulk/cocpit/cpi_data/training_da...,0.686028,0.654362,260.795628,322578.5,2705.948265,0.665657,0.460028,0.553613,0.882135,0.008388,0.960147,640.874171,21.106886
1,compact_irreg,/home/vanessa/hulk/cocpit/cpi_data/training_da...,0.871030,0.873343,328.507515,525003.5,3957.369619,0.669537,0.590304,0.421268,0.818547,0.007538,0.889839,817.591106,9.312568
2,compact_irreg,/home/vanessa/hulk/cocpit/cpi_data/training_da...,0.970808,0.925908,300.808909,492045.0,3625.362479,0.759562,0.570443,0.470448,0.869063,0.007368,0.907938,791.511940,29.683726
3,compact_irreg,/home/vanessa/hulk/cocpit/cpi_data/training_da...,0.937285,0.694526,275.595265,367257.0,3029.019333,0.565226,0.467579,0.503009,0.811227,0.008248,0.933003,683.817326,10.816930
4,compact_irreg,/home/vanessa/hulk/cocpit/cpi_data/training_da...,0.923469,0.746467,293.537476,397303.5,3310.468036,0.633394,0.544481,0.455568,0.804740,0.008332,0.880249,711.240133,53.601586


In [12]:
emb_path = '/home/jko/ssl-cpi-analysis/data/from-hari-v3/clean_combined_campaign_features.pth'
embeddings = torch.load(emb_path)
emb_np = embeddings.numpy() if isinstance(embeddings, torch.Tensor) else np.array(embeddings)
print(emb_np.shape)

(524033, 384)


In [4]:
# load embeddings
save_dir = '/home/jko/ssl-cpi-analysis/data/from-hari-v3'
save_filename = 'cls_env_all_merged.parquet'
save_path = os.path.join(save_dir, save_filename)
df = pd.read_parquet(save_path)
df.head()

,0,1,2,3,4,5,6,7,8,9,...,Altitude [m],Pressure [hPa],Temperature [C],Ice Water Content [g/m3],PSD IWC [g/m3],concentration ratio,area ratio,mass ratio,Campaign,Perimeter [pixels]
0,1.710554,1.393568,3.177823,-0.757673,-0.633948,-0.168053,-1.560423,-1.539199,1.107319,3.162958,...,8238.878125,343.835016,-39.222314,0.041004,0.059899,0.000004,0.000129,0.000522,ARM,NaN
1,2.639770,1.870557,3.738529,-1.303642,-0.456105,0.182570,-1.645625,-1.389037,0.564212,2.378380,...,8234.885352,344.035004,-42.010040,0.027216,0.055219,0.000453,0.012640,0.018208,ARM,NaN
2,-0.739239,-0.054968,0.947078,-0.589259,1.388247,-0.067568,-1.092112,0.602213,0.591173,0.340691,...,7612.743652,376.370209,-36.774002,0.046460,0.086047,0.000000,0.000000,0.000000,ARM,NaN
3,1.567777,1.446529,2.506802,-1.067740,0.357187,0.265005,-0.560557,-1.382249,1.367496,2.629955,...,9439.850781,287.800226,-49.255927,0.027244,0.075817,0.000000,0.000000,0.000000,ARM,NaN
4,2.114330,0.976239,3.096801,-1.177441,0.075485,0.933317,-1.363156,-1.865664,0.227272,2.527692,...,7919.968848,360.108545,-39.378160,0.010284,0.017845,0.000000,0.000000,0.000000,ARM,NaN


In [5]:
df_test['filename'] = df_test['filename'].apply(os.path.basename)
df_test['filename'].head()

0        2016_1014_022252_52.png
1     2004_1006_203416_99_10.png
2    2004_1017_210056_693_56.png
3        2016_1018_023545_68.png
4    2004_1006_212858_501_16.png
Name: filename, dtype: object

In [6]:
num_matches = df_test['filename'].isin(df['filename']).sum()
print(num_matches)

1337


In [10]:
df['filename'].head()

0    2000_0309_200811_599_3.png
1    2000_0313_200622_504_9.png
2    2000_0313_211625_849_8.png
3    2000_0309_212316_10_12.png
4    2000_0313_213953_626_9.png
Name: filename, dtype: object

In [7]:
df_merged = pd.merge(df, df_test, on='filename')
df_merged.shape

(1337, 446)

In [ ]:
print(list(df_merged.columns))

In [ ]:
df_merged['classification'].unique()

In [ ]:
train[['Classification', 'classification', 'filename']].head(10)

In [ ]:
# get random balanced subset of 200 per each class
samples_per_class = 200
indices = []

for cls in df_merged['classification'].unique():
    idx = df_merged[df_merged['classification'] == cls].sample(n=samples_per_class, random_state=42).index.tolist()
    indices.extend(idx)

test = df_merged.iloc[indices]
test.shape

In [ ]:
train_indices = df_merged.index.difference(indices).tolist()
train = df_merged.iloc[train_indices]
train.shape

Note: classification and Classification columns are different.
- classification (lower-case): is the hand-labeled from 21k dataset
- Classification (upper-case): is the pseudo labels from final VGG16 model from Przybylo et al.

In [ ]:
# Logistic regression on 21K labeled subset
X_cols = [str(i) for i in range(384)]
X_train, y_train = train[X_cols], train['classification']
X_test, y_test = test[X_cols], test['classification']
# scale data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
# Train logistic regression
clf = LogisticRegression()
clf.fit(X_train_scaled, y_train)
# Predict
y_pred = clf.predict(X_test_scaled)
# Print classification report (zero_division=0 to silence warning)
print(classification_report(y_test, y_pred, zero_division=0))

In [ ]:
# Compute and plot confusion matrix
cm = confusion_matrix(y_test, y_pred, labels=clf.classes_, normalize='true')
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=clf.classes_)
fig, ax = plt.subplots(figsize=(8, 8))
disp.plot(ax=ax, cmap='Blues', xticks_rotation=45)
plt.title("Logistic Regression Confusion Matrix")
plt.tight_layout()
plt.show()